# LandRisk AI: Phase 2 EDA (cleaned version)

**Goal:** test my hypotheses against the data with numbers, and decide what carries into modelling.
**Reminder:** the data is simulated, so confirming a hypothesis here confirms the simulator and the pipeline, not real-world behaviour.

**Hypotheses before plots**

**Hypothesis 1 — Something you expect to matter**

*physical progress vs time already consumed* -
 For example:
- Project has consumed 80% of its planned time
- But physical progress is only 40%
That looks suspicious because the project is behind its expected execution pace.
I would phrase your hypothesis as:
Projects that have consumed a large proportion of their planned duration but achieved relatively low physical progress are expected to have higher delay risk.

**Hypothesis 2 — Land acquisition**

Let say:
45% land acquired → 55% still unresolved → potentially more risk
95% land acquired → only 5% remaining → potentially less risk

The important thing here is that 40% and 95% are just examples. The dataset contains many values, so the actual EDA should determine whether the pattern exists.
Better hypothesis:
Projects with lower land acquisition completion are expected to have higher delay risk than projects with higher land acquisition completion.

**Hypothesis 3 — Spending**

physical progress = 40% , expenditure = 75% versus physical progress = 40% , expenditure = 45%
Development projects expenditure involves planning, allocating, and managing financial resources to achieve project objectives efficiently.
So let say a project has spend that expenditure 75% but its physical progress is only done around 40...that seem like it  planning, allocating, and managing financial resources to achieve project objective is not manage in fine way that a project that under goes having physical progress = 40% , expenditure = 45%

Projects where expenditure is substantially higher than physical progress are expected to have higher delay risk.

**Hypothesis 4 — Correlation**

Projects with lower physical progress are expected to have higher delay risk.
Projects with both low physical progress and low land acquisition completion are expected to show higher delay risk.

**Hypothesis 5 — Something you expect not to matter**

State is expected to have little or no meaningful relationship with delay risk compared with project execution factors such as physical progress and land acquisition.

##  Hypotheses (written before looking at the data)

Each one says what I expect **and what result would count against it**.

| # | Hypothesis | Would be contradicted if... |
|---|---|---|
| H1 | Projects that are behind schedule (progress well below time consumed) have a higher delay rate | The delay rate is flat or falls as schedule slippage worsens |
| H2 | Lower land acquisition completion means a higher delay rate (among projects that need land) | The delay rate does not rise as land acquired falls |
| H3 | Expenditure far ahead of physical progress means a higher delay rate | The delay rate is flat across spending-progress gap bins |
| H4 | Low physical progress **and** low land acquisition together give a higher delay rate than either alone | The corner cell is no worse than the edges |
| H5 | State has little or no relationship with delay risk | A statistical test shows a clear relationship |

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

# Loading Dataset
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()

DATA_PATH = NOTEBOOK_DIR.parent/"data"/"infrastructure_projects.csv"
data = pd.read_csv(DATA_PATH)
print(f"Dataset Shape:- {data.shape}")
data.head(3)

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.describe(include=object).T

In [ ]:
data.isna().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.nunique()

## 1. Data audit

In [ ]:
print("shape:", data.shape)
data.dtypes

In [ ]:
# Missing values: count AND percentage, only for columns that have any
missing = data.isna().sum().to_frame("missing")
missing["pct"] = 100 * missing["missing"] / len(data)
missing[missing["missing"] > 0]

In [ ]:
# Target balance, and what a model that ALWAYS predicts "0" would score
share = data["delay_risk"].mean()
print(f"Delayed share: {share:.1%}")
print(f"Accuracy of 'always predict 0': {1 - share:.1%}")

**Columns that must not be model features** (answer in your own words in the cell below):
`project_id`, `project_name`, `delay_risk` (it is the target), and anything from `hidden_truth.csv`.

In [ ]:
# Continuous columns only: skew is meaningless for binary (0/1) columns like the target
CONTINUOUS = [
    "project_cost_cr", "planned_duration_months", "months_elapsed",
    "physical_progress_pct", "expenditure_pct", "contractor_experience_years",
    "land_acquisition_pct",
]

def describe_skew(value, tol=0.5):
    if abs(value) < tol:
        return "roughly symmetric"
    return "right-skewed" if value > 0 else "left-skewed"

skew_table = pd.DataFrame({"skew": data[CONTINUOUS].apply(lambda s: stats.skew(s, nan_policy="omit"))})
skew_table["shape"] = skew_table["skew"].apply(describe_skew)
skew_table


## 2. Feature engineering

In [ ]:
# Time already consumed, as a percentage of the plan (0-100)
data["time_elapsed_pct"] = 100 * data["months_elapsed"] / data["planned_duration_months"]

# Progress minus time. Negative = BEHIND schedule, positive = ahead (matches DATA_CARD)
data["schedule_slippage"] = data["physical_progress_pct"] - data["time_elapsed_pct"]

# Spending minus progress. Positive = money spent ahead of the work
data["spending_progress_gap"] = data["expenditure_pct"] - data["physical_progress_pct"]

# Land still to acquire (0 for projects that need no land, by the D-007 convention)
data["land_gap"] = 100 - data["land_acquisition_pct"]


data.head()


## 3. Distributions (numbers first, then pictures)

In [ ]:
data[CONTINUOUS].describe()

In [ ]:
# Log of cost tames the strong right skew
data["log_cost"] = np.log(data["project_cost_cr"])

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), CONTINUOUS + ["log_cost"]):
    values = data[col].dropna()
    ax.hist(values, bins=30, color="skyblue", edgecolor="white")
    ax.axvline(values.median(), color="blue", linestyle="--", label=f"median {values.median():.1f}")
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. H1 and H3: pace and spending (scatter plots with a reference line)

In [ ]:
palette = {0: "#4C72B0", 1: "#C44E52"}

def scatter_with_diagonal(x, y, xlabel, ylabel, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=data, x=x, y=y, hue="delay_risk", palette=palette, alpha=0.5, ax=ax)
    ax.plot([0, 100], [0, 100], "k--", linewidth=1, label="y = x")   # reference: 'exactly on pace'
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(title="Delay risk (0 = low, 1 = high)")
    plt.tight_layout(); plt.show()

scatter_with_diagonal("time_elapsed_pct", 
                    "physical_progress_pct",
                    "Time consumed (%)", "Physical progress (%)",
                    "Time consumed vs physical progress (below the line = behind schedule)")

scatter_with_diagonal("physical_progress_pct", 
                    "expenditure_pct",
                    "Physical progress (%)", 
                    "Expenditure (%)",
                    "Progress vs expenditure (above the line = spending ahead of work)")

The Above Two Graph reveal the Two relationship in Our Data...
1) **Time consumed vs physical progress** (data points which is below the line = are the project which is behind there decided schedule and are going to have high risk to be delay..)
2) **Progress vs expenditure** ( data points which is above the line are the project data whose finance spending ahead of there current work (Physical progress))

The Time consumed vs physical progress graph demonstrate that the 
- X axis is the Time Consumed by the project when the snapshot was taken.
- Y axis is the Physical progress done uptil now about that individual project.
- Dotted line is an boundary that separate the project that have high risk of delay and project that have low risk of delay.
- The Project below the line (Red points) are the projects which is behind schedule or which has high risk of been delay..
- The Project above the line (blue points) are the projects which is ahead/with schedule or which has low risk of been delay..


The Physical Progress vs expenditure graph demonstrate that the 
- X axis is the Physical Progress by the project when the snapshot was taken.
- Y axis is the expenditure pct uptil now about that individual project.
- Dotted Line separate the relationship, where the project above the dootted line are the projects which is ahead of there work which mean there spend expenditure is seffically spend with perfect physical work done...

### Risk curves: how does the delay rate change across a driver?

Bin a driver, compute the delay rate in each bin, and **show the bin sizes** (a rate from 20 projects is not a rate from 300).

In [ ]:
BASELINE = data["delay_risk"].mean()

def risk_curve(df, col, bins=None, q=None, title=None, xlabel=None):
    # q -> equal-count bins (qcut); bins -> fixed edges (cut)
    if q is not None:
        binned = pd.qcut(df[col], q=q, duplicates="drop")
    else:
        binned = pd.cut(df[col], bins=bins, include_lowest=True)
    table = (df.groupby(binned, observed=True)["delay_risk"]
               .agg(projects="size", delay_rate="mean"))
    table["delay_rate"] = 100 * table["delay_rate"]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars = ax.bar(table.index.astype(str), table["delay_rate"], color="#4C72B0")
    ax.axhline(100 * df["delay_risk"].mean(), color="red", linestyle="--", label="group average")
    for bar, n in zip(bars, table["projects"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"n={n}", ha="center", fontsize=8)
    ax.set_ylabel("Delay rate (%)"); ax.set_xlabel(xlabel or col)
    ax.set_title(title or f"Delay rate by {col}")
    ax.legend(); plt.xticks(rotation=25); plt.tight_layout(); plt.show()
    return table

slip_table = risk_curve(data, "schedule_slippage", q=6,
                        title="H1: delay rate by schedule slippage (negative = behind schedule)")
slip_table

**The schedule_slippage mean the Schedule time or Project Schedule time - Phyiscal progress done...**
- It demonstrate the if the schedule_slippage is Negative the delay risk of that project is going to be high cause that particular project phyiscal progress is behind there Project time to complete the project which, will going to cause the project to be delay..
- If schedule_slippage is Positve the delay risk of that project is going to be low cause that particular project phyiscal progress is within there Project time to complete the project, which will have less chance that the project to be delay..

1) Graph Demonstration - The Bar which is below the Dotted line show to have less delay rate where opposite also True..

In [ ]:
gap_table = risk_curve(data.dropna(subset=["spending_progress_gap"]), "spending_progress_gap", q=6,
                       title="H3: delay rate by spending-progress gap (positive = spending ahead)")
gap_table

## 5. H2: land acquisition (only projects that need land)

Projects that need **no** land are stored as exactly 100% acquired (decision D-007). Mixing them in
creates a big artificial pile at 100 and hides the real pattern. So analyse them separately.

In [ ]:
needs_land = data[data["land_acquisition_required"] == 1].copy()
print("Projects needing land:", len(needs_land), "of", len(data))
print("Exactly 100% acquired, all projects:", (data["land_acquisition_pct"] == 100).sum())
print("Exactly 100% acquired, among those needing land:", (needs_land["land_acquisition_pct"] == 100).sum())
print("Delay rate: needs land =", f"{needs_land['delay_risk'].mean():.1%}",
      "| needs no land =", f"{data.loc[data['land_acquisition_required'] == 0, 'delay_risk'].mean():.1%}")

In [ ]:
land_table = risk_curve(needs_land, "land_acquisition_pct", bins=[0, 40, 60, 80, 99.9, 100],
                        title="H2: delay rate by land acquired (projects that need land)")
land_table

delay rate by land acquired (projects that need land) - This tell that project how land acquired is still remaining in significant level that particular project show High risk of being delay...

About the graph:- 
1) Project which land_acquired_pct is high around (80, 99.9] and (99.9,100] those project have very low chance of project being delay..
2) Project which land_acquired_pct is low around (-0.001, 40.0] and (40.0, 60.0] those project have very high chance or high delay rate of project being delay..

## 6. Contractor experience and sector (the drivers I did not hypothesise about)

In [ ]:
exp_table = risk_curve(data.dropna(subset=["contractor_experience_years"]),
                       "contractor_experience_years", bins=[0, 5, 10, 15, 20, 30],
                       title="Delay rate by contractor experience (years)")
exp_table

delay rate by contractor_experience_years (projects whose contractor_experience_years matter) - This tell that project whose contractor_experience_years is still remaining in significant year that particular project show low risk of being delay...

About the graph:- 
1) Project which contractor_experience_years is high around (20, 30] and (15.0, 20.0] those project have very low chance of project being delay.. and that particular project bar comes below the dotted line, where the dotted line separate the classification of project have high / low rate of being delay.

2) Project which contractor_experience_years is low around (-0.001, 5.0] and (5.0, 10.0] those project have very high chance or high delay rate of project being delay..cause that contractor experience is not as good as contractor whose experience is around 20-30

## 8. H5: state (and sector for comparison)

A rate from a small group is noisy. Use the standard error and a chi-square test instead of eyeballing bars.

In [ ]:
# State vs delay risk

state_risk = (
    data.groupby("state")["delay_risk"].mean().sort_values(ascending=False)
    * 100
)


In [ ]:
plt.figure(figsize=(10, 6))

state_risk.sort_values().plot(kind="barh")

plt.xlabel("High-Risk of Projects Going to be Delay (%)")
plt.ylabel("State")
plt.title("Delay Risk Rate by State")

plt.tight_layout()
plt.show()

In [ ]:
sector_risk = (
    data.groupby("sector")["delay_risk"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

In [ ]:
plt.figure(figsize=(10, 7))

sector_risk.sort_values().plot(kind="barh")

plt.xlabel("High-Risk of Projects going to be Delay (%)")
plt.ylabel("Sector")
plt.title("Delay Risk Rate by Sector")
plt.tight_layout()
plt.show()

## 9. Correlations

In [ ]:
CORR_COLS = CONTINUOUS + ["time_elapsed_pct", "schedule_slippage", "spending_progress_gap", "land_gap", "land_acquisition_required", "delay_risk"]
corr = data[CORR_COLS].corr()    

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", annot_kws={"size": 8})
plt.title("Correlation matrix")
plt.tight_layout(); plt.show()

corr["delay_risk"].drop("delay_risk").sort_values()

## 10. Evidence summary (fill in the last two columns yourself)

In [ ]:
worst_slip, best_slip = slip_table["delay_rate"].iloc[0], slip_table["delay_rate"].iloc[-1]
worst_gap, best_gap = gap_table["delay_rate"].iloc[-1], gap_table["delay_rate"].iloc[0]
low_land, high_land = land_table["delay_rate"].iloc[0], land_table["delay_rate"].iloc[-1]
corner = rate.iloc[0, 0]; best_cell = rate.iloc[-1, -1]
p_state = chi_square(data, "state")[1]; p_sector = chi_square(data, "sector")[1]

evidence = pd.DataFrame([
    ["H1", "Delay rate: most-behind vs most-ahead slippage bin", f"{worst_slip:.0f}% vs {best_slip:.0f}%"],
    ["H2", "Delay rate: land < 40% vs land at 100% (needs land)", f"{low_land:.0f}% vs {high_land:.0f}%"],
    ["H3", "Delay rate: largest vs smallest spending gap bin", f"{worst_gap:.0f}% vs {best_gap:.0f}%"],
    ["H4", "Delay rate: low progress + low land vs high progress + high land", f"{corner:.0f}% vs {best_cell:.0f}%"],
    ["H5", "Chi-square p-value, state (sector for comparison)", f"state p = {p_state:.3f}; sector p = {p_sector:.4f}"],
], columns=["Hypothesis", "Test used", "Number"])
Verdict = ['Supported','Supported','Weakly supported','Supported','Not supported']
evidence["Verdict"] = Verdict
evidence